<h1 align="center">Rwanda Maize–Urea Trial</h1>
<h2 align="center">5. Data Enrichment: Rainfall & Soil</h2>

**Goal:** fill two gaps the brief highlights:
1. **Weather:** there is no weather data. This notebook adds `rain_plant_silking`, total rainfall from planting to silking, from **CHIRPS v2.0** daily rainfall (~5 km cells). It is downloaded from NOAA's ERDDAP server and needs nothing beyond pandas.
2. **Soil:** `tot_N`, `org_C` and `sand` are completely empty. They are filled from **SoilGrids 2.0** (ISRIC, ~250 m cells).

**Running this notebook downloads the data** (internet access required). Every response is cached in `01_Data/external/cache/`, so:
* a second run takes seconds;
* if a run is interrupted, re-running continues where it stopped.

Approximate first-run times:
* **CHIRPS:** a few minutes, one small request per 5 km cell.
* **SoilGrids:** about 45 minutes. ISRIC's REST API allows ~5 requests per minute and the trial has ~220 distinct locations. For a faster route, install `rasterio` and set `SOIL_SOURCE = "vrt"` to read the global rasters directly.

The same code can also be run from a terminal: `python scripts/fetch_chirps_rainfall.py` and `python scripts/fetch_soilgrids.py`.

> **Data note.** The trial data is confidential and is not in this repository. The outputs shown were produced on the real data. If you run this notebook without it, it uses the synthetic sample in `01_Data/sample/` and the numbers will differ.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "02_Notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Image, display

from maize import config, data

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
MODE = config.data_mode()
OUT = config.output_paths(MODE)
print("Data source:", "confidential trial data (01_Data/raw)" 
      if MODE == "raw" else "synthetic sample (01_Data/sample)")

sys.path.insert(0, str(ROOT / "scripts"))
import fetch_chirps_rainfall
import fetch_soilgrids

Data source: synthetic sample (01_Data/sample)


## 1. What needs to be fetched

In [2]:
src = data.load_sources(MODE)
fields = src["field"]
pd.Series({
    "Field-seasons": len(fields),
    "Distinct SoilGrids locations (~100 m)": fields[["lat", "long"]].round(3).drop_duplicates().shape[0],
    "Soil variables 100% empty": ", ".join(c for c in ["tot_N", "org_C", "sand"] if src["soil"][c].isna().all()),
    "Plots needing a rainfall window": len(src["plot"]),
    "Plots without a silking date (window undefined)": src["plot"]["silking_date"].isna().sum(),
}, name="").to_frame()

,
Field-seasons,397
Distinct SoilGrids locations (~100 m),397
Soil variables 100% empty,"tot_N, org_C, sand"
Plots needing a rainfall window,1590
Plots without a silking date (window undefined),9


**Method choices**
* **Rainfall window.** Rainfall is summed daily from planting to silking, inclusive, for the CHIRPS cell containing each field. Plots without a silking date, or with dates out of order, are left missing.
* **SoilGrids depth.** Values are averaged over 0–30 cm, weighted by layer thickness (0–5, 5–15, 15–30 cm), which matches the maize rooting zone where fertiliser is applied. Units: N and SOC in g/kg, sand and clay in %, pH in water.
* **Modelled vs measured.** SoilGrids values go in separate `sg_*` columns. The measured pH is never overwritten.
* **Brief deliverable 5.** `01_Data/external/Soil_PL_filled.csv` is the soil table for every plot, with `tot_N` and `org_C` (%) and `sand` (%) filled, plus a source column.

## 2. Download

Set a flag to `False` to skip a source (for example, once its file exists).

In [3]:
FETCH_RAINFALL = True
FETCH_SOIL = True
SOIL_SOURCE = "rest"   # "rest" (no extra packages, ~45 min first time) or "vrt" (needs rasterio, faster)

In [6]:
if FETCH_RAINFALL:
    try:
        rain = fetch_chirps_rainfall.run(source="erddap", mode=MODE)
        display(rain["rain_plant_silking"].describe().to_frame("rain_plant_silking (mm)").T)
    except Exception as e:
        print("CHIRPS download failed:", e)
        print("Check the internet connection, or try fetch_chirps_rainfall.run(source='iridl') "
              "(needs: pip install xarray netCDF4).")

CHIRPS: 1582 plots, 182 grid cells, 2020-08-31 to 2022-06-30 (source: erddap)
CHIRPS download failed: Download failed after 3 attempts: https://coastwatch.pfeg.noaa.gov/erddap/griddap/chirps20GlobalDailyP05.csv?precip[(2020-08-31T00:00:00Z):1:(2022-06-30T00:00:00Z)][(-2.475)][(29.925)]
<urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>
Check the internet connection, or try fetch_chirps_rainfall.run(source='iridl') (needs: pip install xarray netCDF4).


In [10]:
if FETCH_SOIL:
    try:
        soilgrids = fetch_soilgrids.run(source=SOIL_SOURCE, mode=MODE)
        display(soilgrids.filter(like="sg_").describe().T)
    except Exception as e:
        print("SoilGrids download failed:", e)
        print("Progress so far is cached - re-run this cell to continue, "
              "or set SOIL_SOURCE = 'vrt' (needs: pip install rasterio).")

SoilGrids: 397 field-seasons, 397 locations, 269 not yet cached (about 54 min at the API's rate limit)
  SoilGrids: 10/397 locations
  SoilGrids: 20/397 locations
  SoilGrids: 30/397 locations
  SoilGrids: 40/397 locations
  SoilGrids: 50/397 locations
  SoilGrids: 60/397 locations
  SoilGrids: 70/397 locations
  SoilGrids: 80/397 locations
  SoilGrids: 90/397 locations
  SoilGrids: 100/397 locations
  SoilGrids: 110/397 locations
  SoilGrids: 120/397 locations
  SoilGrids: 130/397 locations
  SoilGrids: 140/397 locations
  SoilGrids: 150/397 locations
  SoilGrids: 160/397 locations
  SoilGrids: 170/397 locations
  SoilGrids: 180/397 locations
  SoilGrids: 190/397 locations
  SoilGrids: 200/397 locations
  SoilGrids: 210/397 locations
  SoilGrids: 220/397 locations
  SoilGrids: 230/397 locations
  SoilGrids: 240/397 locations
  SoilGrids: 250/397 locations
  SoilGrids: 260/397 locations
  SoilGrids: 270/397 locations
  SoilGrids: 280/397 locations
  SoilGrids: 290/397 locations
  SoilG

,count,mean,std,min,25%,50%,75%,max
sg_clay_pct,373.0,35.529,4.869,18.850,33.317,35.917,38.517,46.633
sg_nitrogen_g_kg,373.0,2.593,1.020,1.550,2.000,2.380,2.808,10.190
sg_ph_h2o,373.0,5.538,0.365,4.700,5.300,5.600,5.800,6.800
sg_sand_pct,373.0,38.291,6.532,19.683,33.667,39.617,42.533,54.633
sg_soc_g_kg,373.0,37.523,17.758,18.733,24.350,31.483,44.333,126.783


## 3. Status of the enrichment files

In [11]:
external = data.load_external()
pd.Series({
    "CHIRPS rainfall (chirps_rain_plant_silking.csv)": "rainfall" in external,
    "SoilGrids soil (soilgrids_soil.csv)": "soilgrids" in external,
    "Filled soil table (Soil_PL_filled.csv)": (config.EXTERNAL_DIR / "Soil_PL_filled.csv").exists(),
}, name="available").to_frame()

,available
CHIRPS rainfall (chirps_rain_plant_silking.csv),False
SoilGrids soil (soilgrids_soil.csv),True
Filled soil table (Soil_PL_filled.csv),True


## 4. Explore the added variables

In [12]:
_, combined = data.build_combined(MODE)
added = [c for c in combined.columns if c.startswith(("sg_", "rain_"))]
if not added:
    print("No enrichment data yet - check the messages in section 2.")
else:
    ok = combined[~combined["flag_implausible_yield"]]
    print("Coverage (share of plots with a value):")
    display(ok[added].notna().mean().to_frame("coverage").T)
    display(ok.groupby("meg_env")[added].mean())
    display(ok[added + ["yield_tha"]].corr()["yield_tha"].drop("yield_tha").to_frame("correlation with yield"))

Coverage (share of plots with a value):


,sg_clay_pct,sg_nitrogen_g_kg,sg_ph_h2o,sg_sand_pct,sg_soc_g_kg
coverage,0.935,0.935,0.935,0.935,0.935


,sg_clay_pct,sg_nitrogen_g_kg,sg_ph_h2o,sg_sand_pct,sg_soc_g_kg
meg_env,,,,,
Highlands,32.341,3.190,5.358,35.063,52.252
Low-Mid-Dry,38.872,2.026,5.820,40.963,24.176
Low-Mid-Wet,36.281,2.416,5.477,39.685,32.643
Transitional,35.322,2.547,5.464,38.730,35.964


,correlation with yield
sg_clay_pct,0.134
sg_nitrogen_g_kg,-0.124
sg_ph_h2o,0.143
sg_sand_pct,0.102
sg_soc_g_kg,-0.181


In [13]:
if "rain_plant_silking" in combined:
    import matplotlib.pyplot as plt
    plotting_df = ok.dropna(subset=["rain_plant_silking"])
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, s in enumerate(config.SEASONS):
        d = plotting_df[plotting_df["season"] == s]
        ax.scatter(d["rain_plant_silking"], d["yield_tha"], s=10, alpha=.5, color=config.SEASON_COLORS[s], label=s)
    ax.set_xlabel("Rainfall, planting to silking (mm)")
    ax.set_ylabel("Yield (t/ha)")
    ax.set_title("Yield vs in-season rainfall", loc="left", fontweight="bold")
    ax.legend(title="Season", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.savefig(OUT.figures / "yield_vs_rainfall.png", dpi=160, bbox_inches="tight")
    plt.show()

## What happens next

Once the files exist, every later step uses them automatically:
* **Notebook 06** adds `rain_plant_silking` and the `sg_*` variables to the driver-importance ranking.
* **`python run_pipeline.py`** rebuilds the combined dataset, the tables and the figures with the new variables.
* **The audit log** marks the soil and weather items as resolved.

**Next:** notebook 06, yield prediction and drivers.